# Train your own AI-RAN explainer (Colab, free GPU)

Fine-tune a small open model to turn O-RAN/5GC logs into a root-cause diagnosis. No API key, no local GPU.

**Before you start:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Build the dataset locally with `python make_dataset.py --augment` (harder, less memorizable data -> `train.jsonl`, `val.jsonl`).
3. Run the config cell — it mounts Google Drive. Put the two JSONL in the printed `DATA_DIR` (or upload them when the next cell prompts). The trained adapter is saved to Drive (`OUTPUT_DIR`), so it survives Colab disconnects.

The model learns the same `(logs -> diagnosis JSON)` task the API explainer does, so it's a drop-in local replacement.

In [ ]:
!pip -q install -U transformers peft accelerate datasets bitsandbytes

In [ ]:
# --- config + Google Drive storage (so data + adapter survive Colab disconnects) ---
from google.colab import drive
drive.mount("/content/drive")   # standard mount point; files live under /content/drive/MyDrive/

import os
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated, fits a free T4. Try -3B-Instruct for more quality.
MAX_LEN = 2560   # headroom so the assistant JSON (at the end) is never truncated
EPOCHS = 3       # with early stopping below; raise only if val loss is still falling

PROJECT_DIR = "/content/drive/MyDrive/ORAN_Agent"   # your Drive: My Drive/ORAN_Agent
DATA_DIR = os.path.join(PROJECT_DIR, "data")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "explainer-lora")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# --- make sure train.jsonl / val.jsonl are in DATA_DIR (Drive) ---
# First run: upload them here and they're copied into your Drive. Later runs reuse them.
import os, shutil
for fname, dst in (("train.jsonl", TRAIN_FILE), ("val.jsonl", VAL_FILE)):
    if not os.path.exists(dst):
        from google.colab import files
        print(f"{fname} not in DATA_DIR — upload it now:")
        up = files.upload()
        src = fname if os.path.exists(fname) else next(iter(up))
        shutil.move(src, dst)
print("train:", sum(1 for _ in open(TRAIN_FILE)), "| val:", sum(1 for _ in open(VAL_FILE)))

In [ ]:
# --- load tokenizer + 4-bit model + LoRA ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)
# Lower rank (r=8) + higher dropout = less capacity to memorize the templates.
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.1, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

In [ ]:
# --- chat template + COMPLETION-ONLY masking (train on the diagnosis, not the prompt) ---
# The prompt-only text is a token-prefix of the full text, so we mask those tokens to -100;
# only the assistant JSON + its closing <|im_end|> stay supervised -> the model learns BOTH
# to produce the JSON and to STOP. This is the fix for the systemsystem... degeneration.
from datasets import load_dataset
raw = load_dataset("json", data_files={"train": TRAIN_FILE, "val": VAL_FILE})


def tokenize(ex):
    msgs = ex["messages"]
    full = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    full_ids = tok(full, add_special_tokens=False, truncation=True, max_length=MAX_LEN)["input_ids"]
    prompt_ids = tok(prompt, add_special_tokens=False, truncation=True, max_length=MAX_LEN)["input_ids"]
    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100   # mask system+user; supervise only the assistant diagnosis
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

ds = raw.map(tokenize, remove_columns=raw["train"].column_names)
ex0 = ds["train"][0]
print("seq len:", len(ex0["input_ids"]),
      "| supervised tokens (non -100):", sum(1 for x in ex0["labels"] if x != -100))

In [ ]:
# --- train (QLoRA, completion-only loss) with eval + early stopping ---
# Anti-overfit: evaluate on val each epoch, keep the BEST checkpoint (lowest val
# loss), and stop early if val loss stops improving -> we don't ship the last
# (most over-fit) epoch. Watch train loss vs eval loss: a widening gap = overfit.
from transformers import (TrainingArguments, Trainer, DataCollatorForSeq2Seq,
                          EarlyStoppingCallback)
args = TrainingArguments(
    output_dir=OUTPUT_DIR, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type="cosine",
    weight_decay=0.01, fp16=True, logging_steps=10,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none", optim="paged_adamw_8bit")
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"],
                  data_collator=collator,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
# If you see eval_loss bottom out then rise, that's the overfit point — early
# stopping already rolled back to the best checkpoint for you.

In [ ]:
# --- evaluate: root-cause accuracy on held-out val ---
import json, torch
val_raw = [json.loads(l) for l in open(VAL_FILE)]

def extract_json(t):
    i, j = t.find("{"), t.rfind("}")
    if i < 0 or j < 0:
        return None
    try:
        return json.loads(t[i:j+1])
    except Exception:
        return None

def predict(messages_wo_assistant):
    prompt = tok.apply_chat_template(messages_wo_assistant, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=768, do_sample=False,
                             repetition_penalty=1.1, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

n = ok = 0
for ex in val_raw[:20]:
    gold = json.loads(ex["messages"][-1]["content"])["root_cause_component"]
    pred_obj = extract_json(predict(ex["messages"][:-1]))
    pred = (pred_obj or {}).get("root_cause_component", "<parse-fail>")
    hit = pred.strip().lower() == gold.strip().lower()
    n += 1; ok += int(hit)
    print(("PASS" if hit else "FAIL"), "pred=", pred, "| gold=", gold)
print(f"\nroot-cause accuracy: {ok}/{n} ({100*ok/max(n,1):.0f}%)")

In [ ]:
# --- one full prediction, end to end ---
print(predict(val_raw[0]["messages"][:-1])[:1000])

In [ ]:
# --- save the LoRA adapter to Drive (persists across disconnects) ---
model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("saved adapter to", OUTPUT_DIR)
# Reload later:
#   from peft import PeftModel
#   base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
#   model = PeftModel.from_pretrained(base, OUTPUT_DIR)

## Notes & caveats
- Trained on **synthetic** logs -> it diagnoses synthetic-style logs well, but won't automatically generalize to messy real OAI/Open5GS logs. Same demo->real gap as the API path; close it by training on logs captured from induced failures.
- A 1.5B model learns the *format* and *patterns* reliably; it doesn't reason like a frontier model. Bump to 3B (or more data / scenarios) if accuracy is short.
- Want harder, less memorizable data? `python make_dataset.py --augment --train-seeds 80` (distractor noise + time jitter + paraphrases) and re-upload.
- Test generalization to an UNSEEN failure type: `python make_dataset.py --exclude sync_loss` trains on the other 4 and writes `test_sync_loss.jsonl`; after training, score it with `analyze_logs.py` / `eval.py` — a big drop vs in-distribution accuracy is the real overfitting signal.
- This local model is a drop-in for the Claude call: same system prompt, same `(logs -> diagnosis JSON)` contract.